# Сравнение поиска и финальная оценка

Ноутбук сравнивает word TF-IDF и BM25 на одобренных статьях. Запросы разделены на два файла: calibration нужен для выбора метода и порога отказа; holdout — для однократной финальной оценки зафиксированного решения. Не меняйте метод или порог после просмотра holdout. Корпус и файлы разметки должны оставаться локальными.


## 1. Подготовка корпуса

Статьи должны соответствовать схеме из `data/knowledge/README.md`: уникальный ID, ссылка на источник, дата `YYYY-MM-DD`, язык, владелец, лицензия и `status=approved`. BANKING77 не является базой ответов.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

CORPUS_PATH = ROOT / 'data' / 'knowledge' / 'articles.jsonl'
CALIBRATION_PATH = ROOT / 'data' / 'knowledge' / 'queries_calibration.jsonl'
HOLDOUT_PATH = ROOT / 'data' / 'knowledge' / 'queries_holdout.jsonl'
retriever = None
bm25_retriever = None
calibration_examples = None
holdout_examples = None
holdout_retrieval = None
holdout_thresholds = None


In [ ]:
if not CORPUS_PATH.is_file():
    print('Добавьте одобренные статьи в data/knowledge/articles.jsonl.')
else:
    from src.rag.retriever import BM25Retriever, KnowledgeRetriever
    retriever = KnowledgeRetriever.from_jsonl(CORPUS_PATH)
    bm25_retriever = BM25Retriever.from_jsonl(CORPUS_PATH)
    print(f'Документов: {len({chunk.document_id for chunk in retriever.chunks})}')
    print(f'Фрагментов: {len(retriever.chunks)}')


## 2. Ручной просмотр выдачи

Измените запрос на свой. Сначала сравните, какие документы поставили наверх два ранжировщика. Их численные score лежат в разных шкалах и между собой не сравниваются.


In [ ]:
query = 'card payment reversed'
if retriever is not None:
    tfidf_hits = retriever.search(query, top_k=5)
    for rank, hit in enumerate(tfidf_hits, start=1):
        print(f'{rank}. {hit.title} | {hit.source}')
        print(f'   {hit.score:.4f} ({hit.score_type}) | {hit.text}\n')
    bm25_hits = bm25_retriever.search(query, top_k=5)
    for rank, hit in enumerate(bm25_hits, start=1):
        print(f'{rank}. {hit.title} | {hit.source}')
        print(f'   {hit.score:.4f} ({hit.score_type}) | {hit.text}\n')


## 3. Calibration: сравнение и выбор порога

Составьте два независимых файла вручную. Разделяйте близкие переформулировки одного вопроса между группами целиком, чтобы похожие запросы не попали по разные стороны. В каждой строке укажите language; evaluator фильтрует статьи по языку и выводит ranking-метрики отдельно по языкам. В calibration должны быть answerable и unanswerable запросы. В этом разделе сравниваются TF-IDF, BM25 по сетке k1/b и пороги отказа.


In [ ]:
if retriever is None:
    print('Сначала добавьте корпус и выполните предыдущие ячейки.')
elif not CALIBRATION_PATH.is_file():
    print('Добавьте data/knowledge/queries_calibration.jsonl для сравнения методов и выбора порога.')
else:
    from src.rag.evaluation import evaluate_retrieval, load_queries, sweep_abstention_thresholds
    import pandas as pd
    calibration_examples = load_queries(CALIBRATION_PATH)
    tfidf_calibration = evaluate_retrieval(retriever, calibration_examples, k=5)
    bm25_calibration = evaluate_retrieval(bm25_retriever, calibration_examples, k=5)
    calibration_metrics = pd.DataFrame([
        {'method': 'TF-IDF', 'Recall@5': tfidf_calibration['recall_at_5'], 'MRR': tfidf_calibration['mrr']},
        {'method': 'BM25', 'Recall@5': bm25_calibration['recall_at_5'], 'MRR': bm25_calibration['mrr']},
    ])
    display(calibration_metrics)
    tfidf_language_metrics = pd.DataFrame.from_dict(
        tfidf_calibration['language_metrics'], orient='index'
    ).assign(method='TF-IDF')
    bm25_language_metrics = pd.DataFrame.from_dict(
        bm25_calibration['language_metrics'], orient='index'
    ).assign(method='BM25')
    language_metrics = pd.concat(
        [tfidf_language_metrics, bm25_language_metrics], ignore_index=False
    ).rename_axis('language').reset_index()
    display(language_metrics)


### Разбор ошибок на calibration

Таблица показывает запросы, где релевантная статья не попала в top-5 или не стала первой. Сначала сравните найденные ID с разметкой и текстом статьи: ошибка может быть в статье, метке, языке или ранжировании. После просмотра holdout не используйте его случаи для настройки текущей версии; если меняете систему по результатам holdout, понадобится новая независимая выборка.


In [ ]:
if calibration_examples is not None:
    error_rows = []
    for method_name, result in [('TF-IDF', tfidf_calibration), ('BM25', bm25_calibration)]:
        for row in result['answerable_query_results']:
            if not row['top1_is_relevant'] or row['missed_relevant_document_ids_at_k']:
                error_rows.append({
                    'method': method_name,
                    'query': row['query'],
                    'language': row['language'],
                    'relevant_ids': row['relevant_document_ids'],
                    'top1_id': row['top1_document_id'],
                    'top1_score': row['top1_score'],
                    'first_relevant_rank': row['first_relevant_rank'],
                    'top5_ids': row['retrieved_document_ids_at_k'],
                    'missed_relevant_ids': row['missed_relevant_document_ids_at_k'],
                })
    error_table = pd.DataFrame(error_rows)
    if error_table.empty:
        print('На answerable calibration-запросах ошибок top-1/top-5 не найдено.')
    else:
        display(error_table.sort_values(['method', 'language', 'first_relevant_rank'], na_position='last'))


In [ ]:
if calibration_examples is not None:
    bm25_grid_results = []
    for candidate_k1 in [1.0, 1.2, 1.5, 2.0]:
        for candidate_b in [0.3, 0.5, 0.75, 0.9]:
            candidate = bm25_retriever.with_parameters(candidate_k1, candidate_b)
            metrics = evaluate_retrieval(candidate, calibration_examples, k=5)
            bm25_grid_results.append({
                'k1': candidate_k1,
                'b': candidate_b,
                'Recall@5': metrics['recall_at_5'],
                'MRR': metrics['mrr'],
            })
    bm25_grid_table = pd.DataFrame(bm25_grid_results)
    display(bm25_grid_table.sort_values(['MRR', 'Recall@5'], ascending=False))


In [ ]:
SELECTED_METHOD = 'tfidf'
SELECTED_BM25_K1 = 1.5
SELECTED_BM25_B = 0.75
LANGUAGE_CODES = sorted({(example.get('language') or 'unspecified').casefold() for example in calibration_examples or []})
SELECTED_THRESHOLDS = {language: None for language in LANGUAGE_CODES}

if SELECTED_METHOD not in {'tfidf', 'bm25'}:
    raise ValueError("SELECTED_METHOD должен быть 'tfidf' или 'bm25'.")
selected_bm25 = (
    bm25_retriever.with_parameters(SELECTED_BM25_K1, SELECTED_BM25_B)
    if bm25_retriever is not None else None
)
print('Выбранный метод:', SELECTED_METHOD)
print('Параметры BM25:', SELECTED_BM25_K1, SELECTED_BM25_B)
print('Языки calibration:', LANGUAGE_CODES)
print('Пороги по языкам:', SELECTED_THRESHOLDS)


После просмотра calibration вручную зафиксируйте метод, параметры BM25 (если выбран он) и отдельный порог для каждого языка ниже. Пороги между языками и между TF-IDF/BM25 переносить нельзя. Не выбирайте метод, параметры или порог по holdout-результату.


In [ ]:
if calibration_examples is not None:
    tfidf_thresholds = pd.DataFrame(
        sweep_abstention_thresholds(retriever, calibration_examples)
    ).assign(method='TF-IDF')
    bm25_thresholds = pd.DataFrame(
        sweep_abstention_thresholds(selected_bm25, calibration_examples)
    ).assign(method='BM25')
    threshold_results = pd.concat([tfidf_thresholds, bm25_thresholds], ignore_index=True)
    display(threshold_results.sort_values(
        ['method', 'language', 'supported_answer_f1', 'coverage'], ascending=[True, True, False, False]
    ).groupby(['method', 'language']).head(10))


## 4. Holdout: однократная финальная оценка

Заполните holdout до выбора метода, не просматривайте его метрики до фиксации метода и порога. После оценки не возвращайтесь к подбору на этом наборе. При изменении ранжировщика или порога понадобится новый независимый holdout.


In [ ]:
if retriever is None:
    print('Сначала добавьте корпус и выполните предыдущие ячейки.')
elif not HOLDOUT_PATH.is_file():
    print('Holdout отсутствует. Финальную метрику пока не считаем.')
elif not SELECTED_THRESHOLDS or any(value is None for value in SELECTED_THRESHOLDS.values()):
    print('Вручную задайте порог calibration для каждого языка в SELECTED_THRESHOLDS.')
elif retriever is not None:
    from src.rag.evaluation import evaluate_retrieval, load_queries, sweep_abstention_thresholds
    import pandas as pd
    holdout_examples = load_queries(HOLDOUT_PATH)
    selected_retriever = retriever if SELECTED_METHOD == 'tfidf' else selected_bm25
    holdout_retrieval = evaluate_retrieval(selected_retriever, holdout_examples, k=5)
    holdout_thresholds = sweep_abstention_thresholds(
        selected_retriever, holdout_examples, thresholds=SELECTED_THRESHOLDS
    )
    display(pd.DataFrame.from_dict(holdout_retrieval['language_metrics'], orient='index').rename_axis('language'))
    display(pd.DataFrame(holdout_thresholds))
    print({
        'method': SELECTED_METHOD,
        'thresholds_by_language': SELECTED_THRESHOLDS,
        'recall_at_5': holdout_retrieval['recall_at_5'],
        'mrr': holdout_retrieval['mrr'],
    })


## 5. Экспорт калиброванной конфигурации

После расчёта holdout ноутбук может записать candidate config в `work/retrieval_config_candidate.json`. В manifest сохраняются SHA-256 корпуса и файлов разметки, метод, chunking и параметры BM25. Перед использованием сравните конфигурацию, убедитесь, что holdout был просмотрен только после фиксации порогов, затем перенесите поля в `configs/retrieval.json`. Не публикуйте сами статьи и запросы.


In [ ]:
if holdout_retrieval is None or not SELECTED_THRESHOLDS or any(value is None for value in SELECTED_THRESHOLDS.values()):
    print('Candidate config не создан: сначала завершите оценку на holdout и зафиксируйте пороги.')
else:
    import hashlib
    import json
    base_config = json.loads((ROOT / 'configs' / 'retrieval.json').read_text(encoding='utf-8'))
    selected_bm25_config = {'k1': SELECTED_BM25_K1, 'b': SELECTED_BM25_B}
    candidate_config = {
        'method': SELECTED_METHOD,
        'chunking': base_config['chunking'],
        'bm25': selected_bm25_config,
        'thresholds_by_language': SELECTED_THRESHOLDS,
        'selection_status': 'calibrated',
        'calibration_manifest': {
            'corpus_sha256': hashlib.sha256(CORPUS_PATH.read_bytes()).hexdigest(),
            'calibration_queries_sha256': hashlib.sha256(CALIBRATION_PATH.read_bytes()).hexdigest(),
            'holdout_queries_sha256': hashlib.sha256(HOLDOUT_PATH.read_bytes()).hexdigest(),
            'method': SELECTED_METHOD,
            'chunking': base_config['chunking'],
            'bm25': selected_bm25_config,
        },
    }
    candidate_path = ROOT / 'work' / 'retrieval_config_candidate.json'
    candidate_path.parent.mkdir(parents=True, exist_ok=True)
    candidate_path.write_text(json.dumps(candidate_config, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    print(f'Кандидат сохранён: {candidate_path}')


## 6. Сохранение результатов запуска

После завершения holdout можно сохранить сводные метрики, настройки, версии библиотек и SHA-256 входных файлов в отдельную папку `work/retrieval_runs/`. CSV с разбором ошибок содержит тексты запросов, поэтому все артефакты остаются локальными и не добавляются в Git. Сохранённый holdout нужен для аудита уже проведённой оценки, а не для повторной настройки.


In [ ]:
if calibration_examples is None or holdout_retrieval is None:
    print('Артефакты не сохранены: сначала завершите calibration и однократную оценку holdout.')
else:
    from datetime import datetime, timezone
    from importlib.metadata import PackageNotFoundError, version
    import hashlib
    import json
    import platform
    import subprocess
    run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    run_dir = ROOT / 'work' / 'retrieval_runs' / run_id
    run_dir.mkdir(parents=True, exist_ok=False)
    def sha256_file(path):
        return hashlib.sha256(path.read_bytes()).hexdigest()
    try:
        git_revision = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=ROOT, capture_output=True, text=True, check=True).stdout.strip()
        git_worktree_clean = not subprocess.run(['git', 'status', '--porcelain'], cwd=ROOT, capture_output=True, text=True, check=True).stdout.strip()
        git_diff = subprocess.run(['git', 'diff', 'HEAD', '--binary'], cwd=ROOT, capture_output=True, text=True, check=True).stdout
        git_diff_sha256 = hashlib.sha256(git_diff.encode('utf-8')).hexdigest()
    except (FileNotFoundError, subprocess.CalledProcessError):
        git_revision = None
        git_worktree_clean = None
        git_diff_sha256 = None
    package_versions = {}
    for package in ['numpy', 'pandas', 'scikit-learn']:
        try:
            package_versions[package] = version(package)
        except PackageNotFoundError:
            package_versions[package] = None
    run_manifest = {
        'run_id': run_id,
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'git_revision': git_revision,
        'git_worktree_clean': git_worktree_clean,
        'git_diff_sha256': git_diff_sha256,
        'python_version': platform.python_version(),
        'package_versions': package_versions,
        'input_sha256': {
            'corpus': sha256_file(CORPUS_PATH),
            'calibration_queries': sha256_file(CALIBRATION_PATH),
            'holdout_queries': sha256_file(HOLDOUT_PATH),
        },
        'retrieval_config': json.loads((ROOT / 'configs' / 'retrieval.json').read_text(encoding='utf-8')),
        'selection': {
            'method': SELECTED_METHOD,
            'bm25': {'k1': SELECTED_BM25_K1, 'b': SELECTED_BM25_B},
            'thresholds_by_language': SELECTED_THRESHOLDS,
        },
        'holdout_metrics': {
            'recall_at_5': holdout_retrieval['recall_at_5'],
            'mrr': holdout_retrieval['mrr'],
            'query_count': holdout_retrieval['query_count'],
            'answerable_query_count': holdout_retrieval['answerable_query_count'],
            'unanswerable_query_count': holdout_retrieval['unanswerable_query_count'],
        },
    }
    (run_dir / 'run_manifest.json').write_text(json.dumps(run_manifest, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    calibration_metrics.to_csv(run_dir / 'calibration_method_metrics.csv', index=False)
    language_metrics.to_csv(run_dir / 'calibration_language_metrics.csv', index=False)
    error_table.to_csv(run_dir / 'calibration_error_cases.csv', index=False)
    threshold_results.to_csv(run_dir / 'calibration_threshold_sweep.csv', index=False)
    bm25_grid_table.to_csv(run_dir / 'bm25_grid.csv', index=False)
    pd.DataFrame.from_dict(holdout_retrieval['language_metrics'], orient='index').rename_axis('language').reset_index().to_csv(run_dir / 'holdout_language_metrics.csv', index=False)
    pd.DataFrame(holdout_thresholds).to_csv(run_dir / 'holdout_threshold_metrics.csv', index=False)
    print(f'Локальные артефакты запуска сохранены: {run_dir}')


## Интерпретация

Recall@5 и MRR оценивают ранжирование статей. Coverage и метрики supported answers оценивают принятие/отказ при закреплённом пороге. Таблица language_metrics показывает Recall@5/MRR отдельно по каждому language. Пороговые кривые строятся отдельно по языкам, так как распределения score могут различаться. Score не является вероятностью. Численные результаты появятся только после добавления реального корпуса и экспертной разметки обоих наборов. Подробности протокола и ограничений записаны в Word-пособии и `reports/research.md`.
